# GUV Image Analysis Pipeline

This notebook contains a pipeline for analyzing GUV (Giant Unilamellar Vesicle) images. It processes both single-frame and time-trace data from .tif files.

## Imports and Setup

In [ ]:
import csv
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from scipy.interpolate import make_interp_spline

# Import custom modules
from A00_init import get_exps

## GUV Class Definition

In [ ]:
class GUV:
    def __init__(self):
        self.exp_id = 0
        self.label = 'any_label'
        self.comment = []
        self.use_it = []
        self.crop_it = []
        self.dt = 1

## Helper Functions

### `get_data_selections(run_id)`

This function reads GUV data from an Excel file and returns a list of GUV objects for a specific run ID.
We define a `GUV` class to store properties of a single GUV movie.

In [ ]:
def get_data_selections(run_id):
    wb = load_workbook(filename="data_overview.xlsx")
    sheet_files = wb['guvs']
    
    # Create a dictionary of column names
    Header = {COL[0].value: idx for idx, COL in enumerate(sheet_files.iter_cols(1, sheet_files.max_column))}
    
    Guv_list = []
    for row_cells in sheet_files.iter_rows(min_row=2, max_row=sheet_files.max_row):
        Guv = GUV()
        Guv.exp_id = row_cells[Header["exp_id"]].value
        Guv.movie_id = row_cells[Header["movie_id"]].value
        Guv.label = row_cells[Header["guv_label"]].value
        Guv.use_it = row_cells[Header["use"]].value
        Guv.crop_it = row_cells[Header["crop"]].value
        Guv.dt = row_cells[Header["dt(s)"]].value
        if Guv.exp_id == run_id:
            Guv_list.append(Guv)
    return Guv_list

## Main Analysis Pipeline

Set up the experiment parameters:

In [ ]:
expi = 5.2  # 2: flexibles; 3: less_challenging ones 4: single-image tiffs
movie_to_use_list = [462, 463, 465, 466]
movie_simbol_list = ['r', 'k', 'm', 'b']

initval = get_exps(expi)

### Single-frame Data Processing

If the data is single-frame .tif files, we collect all data points and save them in one file.

In [ ]:
if initval.suffix == '.tif' and initval.sequence == 'single_frame':
    load_dirname = initval.mainpath_out + initval.subdir + "/A20b_processed/"
    csv_path_in = Path(load_dirname)
    
    # Collect data from all CSV files
    name, data = [], []
    for csv_source in csv_path_in.glob("**/*.csv"):
        print(csv_source.stem)
        with open(csv_source) as f:
            reader = csv.DictReader(f, delimiter=";")
            for row in reader:
                name.append(csv_source.stem)
                data.append(row)
    
    # Save collected data to a new CSV file
    csv_path_out = Path(initval.mainpath_out + initval.subdir + "/A30_processed/")
    csv_path_out.mkdir(exist_ok=True)
    
    csv_target = load_dirname + "collected_data.csv"
    with open(csv_target, "w", newline='') as csv_f:
        writer = csv.writer(csv_f, delimiter=";")
        writer.writerow(data[0].keys())
        for data_row in data:
            writer.writerow(data_row.values())

### Time-trace Data Processing and Visualization

For time-trace .tif files, we process the data and create visualizations.

In [ ]:
if initval.suffix == '.tif' and initval.sequence == 'time_trace':
    fig, axs = plt.subplots(2, 4, figsize=(20, 15))
    
    Guv_list = get_data_selections(round(expi))
    
    load_dirname_A20a = initval.mainpath_out + initval.subdir + "/A20a_tracked/"
    load_dirname_A20b = initval.mainpath_out + initval.subdir + "/A20b_processed/"
    
    guv_labels = []
    
    for this_guv in Guv_list:
        if this_guv.use_it == 1 and this_guv.movie_id in movie_to_use_list:
            movie_use_index = movie_to_use_list.index(this_guv.movie_id)
            guv_labels.append(this_guv.label)
            
            # Read data from CSV
            csv_source = Path(load_dirname_A20b + this_guv.label + "_all_data.csv")
            with open(csv_source) as g:
                reader = csv.DictReader(g, delimiter=";")
                data = list(reader)
            
            # Process and plot data
            plot_ax, area, roundness, major_minor = [], [], [], []
            c0_edge_mx, c1_edge_mx, c0_edge_sm, c1_edge_sm = [], [], [], []
            c1_std_sum_rel, ratio1 = [], []
            
            for fri, row in enumerate(data):
                # ... (data processing code)
            
            simbol_color = movie_simbol_list[movie_use_index]
            sz = 2
            
            # Plot various metrics
            axs[0, 0].plot(plot_ax, area, 'o-', markersize=sz, color=simbol_color)
            axs[0, 0].set_ylabel("area")
            axs[0, 0].set_title("area")
            
            # ... (more plotting code for other metrics)
    
    plt.tight_layout()
    plt.show()

This completes the GUV image analysis pipeline. The notebook processes both single-frame and time-trace .tif files, collecting data and creating visualizations for various GUV metrics.